In [ ]:
#CANADIAN FIRE WEATHER INDEX SET FOR NEBRASKAN CLIMATE
#REFERENCE TO HERE
##  https://ostrnrcan-dostrncan.canada.ca/entities/publication/d96e56aa-e836-4394-ba29-3afe91c3aa6c
#PATH CALLS AND VARIABLE LISTS HAVE BEEN COMMENTED OUT AND MARKED WITH <<<<<<<<<<<<<<<<<<<<<
#WRITTEN TO BE USED WITH THE EVOFLOOD STATISTICALLY DOWNSCALED DATASET
##  https://catalogue.ceda.ac.uk/uuid/c107618f1db34801bb88a1e927b82317/
try :
    import math as my
except ImportError as e :
    print(f'{e}')
    
try :
    import pandas as pd
    print(f'Pandas {pd.__version__} loaded successfully')
except ImportError as e :
    print(f'{e}')
    
try :
    import numpy as np
    print(f'Numpy {np.__version__} loaded successfully')
except ImportError as e :
    print(f'{e}')
    
try :
    import xarray as xr
    print(f'Xarray {xr.__version__} loaded successfully')
except ImportError as e :
    print(f'{e}')
    
try :
    import matplotlib as mat
    import matplotlib.pyplot as plt
    print(f'Matplotlib {mat.__version__} loaded successfully')
except ImportError as e :
    print(f'{e}')

try :
    import calendar as cally
except ImportError as e :
    print(f'{e}')   

try :
    import time as time
except ImportError as e :
    print(f'{e}')

try :
    import scipy as sp
except ImportError as e :
    print(f'{e}')
    
try :
    import datetime
    from datetime import timedelta
except ImportError as e :
    print(f'{e}')   
    
import os
import sys
pd.set_option('display.max_rows', None)
date = datetime.datetime.now()


global T, RH, W, RA, D1, filepath1, filepath2, model, mode, now, locale, local, ENTIRETHING    

def ENTIRETHING(T, RH, W, RA, DMCtrue, windtrue) :
    
    def drought_code(t, precip, prev) :
    # t = INTERNAL TEMPERATURE VARS, precip = PRECIP VALUE IN PLACE FOR RA
        e = np.e

            #DICTONARY FOR DAYLENGTH VALUES   
        droughtlength = { #THESE ONLY WORK FOR NEBRASKA'S LATITUDE, STATES BELOW NEBRASKA WILL HAVE DIFFERENT VALUES
        1: -1.6, #JAN
        2: -1.6, #FEB
        3: -1.6, #MAR
        4: 0.9, #APR
        5: 3.8, #MAY
        6: 5.8, #JUN
        7: 6.4, #JUL
        8: 5.0, #AUG
        9: 2.4, #SEP
        10: 0.4, #OCT
        11: -1.4, #NOV
        12: -1.6, #DEC
        }

        def monthcall_DC(step): #FUNCTION FOR CHECKING THE MONTH AND RETRIEVING DL
                try:
                    stamp = pd.to_datetime(step.time.values)
                    month_call = step['time'].dt.month.item()
                    month_call = stamp.month        
                    return droughtlength[month_call]
                except Exception as e:
                    print(f"Month Call Error \n {e}")
                    month = 6.4 #JULY VALUE ERR ON SIDE OF CAUTION
                    #IF YOUR DATA RETURNS ABNORMALLY HIGH VALUES THAT SUDDENLY DROP TO EXTREMELY LOW VALUES THEN THE FAILSAFE MOST LIKELY TRIGGERED
                
                    return month

        def evapotranspiration(T) :
            #T = tas, date = ENTIRE DATETIME OBJECT FROM THE netCDF
            t12 = np.maximum(T,-2.8)
            Lf = monthcall_DC(T)
            Vs = 0.36 * (t12 + 2.8) + Lf
            V = np.maximum(Vs, 0.00001)
            return V

        def DCrain(temp, precip, prev) :
            #temp = CURRENT TEMPERATURE, precip = CURRENT PRECIPITATION VALUE, prev = PREVIOUS DAY'S DC
            e = np.e
            rd = (0.83 * precip)-1.27
            Q0 = (800 * (e)**(-prev/400))
                    #FOLLOWING CONVERTS Qr TO Dr READING FROM Q0
            Qr = Q0 + (3.937 * rd)
            Dr = (400 * np.log(800/Qr))
            finalrain = np.maximum(Dr, 0)

            V = evapotranspiration(temp)
            final = finalrain + 0.5 * V
            return final

        def DCdry(temp, prev) :
            #temp = CURRENT TEMPERATURE, prev = PREVIOUS DAY'S DC        
            V = evapotranspiration(temp)
            final = prev + 0.5 * V
            return final

        dcframe = xr.full_like(t, fill_value = np.nan) #THIS IS THE EMPTY ARRAY FOR OUR FINDINGS TO PRINT TO 
        start_time = time.perf_counter()
        for time_x in range(len(t)) :
            t_x = t.isel(time=time_x)
            ra_x = precip.isel(time=time_x)

            raincheck = np.where(ra_x.values >= 2.8,
                                DCrain(t_x, ra_x, prev),
                                DCdry(t_x, prev))
            drought = (np.round(raincheck))
            dcframe.values[time_x] = drought
            prev = dcframe.values[time_x]
        loop_time = timedelta(seconds=time.perf_counter() - start_time)
        print(f"This took us about {loop_time}")
        dcframe = np.round(dcframe)
        dcframe.attrs['units'] = 'degrees'
        DC = dcframe.rename('dc')
        return DC

    dcrunrun = drought_code(T, RA, 15)
    
    dcrunrun.to_netcdf(f'{date} - {locale}_{now}_{mode}_{model}_DC.nc')
    print(f'DC {model} Success! Saved as an .nc of the ({now}) {model} {mode}')
    DC = dcrunrun

    dufflength = {
            1: 6.5, #JAN
            2: 7.5, #FEB
            3: 9, #MAR
            4: 12.8, #APR
            5: 13.9, #MAY
            6: 13.9, #JUN
            7: 12.4, #JUL
            8: 10.9, #AUG
            9: 9.4, #SEP
            10: 8.0, #OCT
            11: 7.0, #NOV
            12: 6.0, #DEC
        }
    def monthcall_DUFF(step): #FUNCTION FOR CHECKING THE MONTH AND RETRIEVING DL
            try:
                month_call = step['time'].dt.month.item()
                month = dufflength[month_call]
                return month
            except Exception as e:
                month = 12.4 #JULY VALUE ERR ON SIDE OF CAUTION
                #IF YOUR DATA RETURNS ABNORMALLY HIGH VALUES THAT SUDDENLY DROP TO EXTREMELY LOW VALUES THEN THE FAILSAFE MOST LIKELY TRIGGERED
                return month



    e = np.e

    def raineffect(RA) :
        #RA = precipitation
        #CFWI MANUAL USED Pe AS EFFECTIVE RAINFALL VARIABLE
        Pe = (0.92 * RA) - 1.27
        return Pe

    def priorduff(DMC) :
        #DMC = PREVIOUS DAY'S DUFF VALUE OR IT IS THE VALUE '6' FOR SPIN-UPS
        #CFWI MANUAL USES Mt-1, SHORTENING TO Mt
        e = np.e
        fraction = DMC / 43.43
        upper = 5.6348 - fraction
        Mt = 20 + (e**upper)
        return Mt


    def Ktest(T, RH, day) :
        #T = tas, RH = hurs, day = time_x

        T = np.maximum(T, -1.1) #THIS IS THE TEMPERATURE TEST, JUST REPHRASED FOR NUMPY
        humidity = (100-RH)
        temp = T + 1.1
        sunlight = monthcall_DUFF(day)
        K = 1.894 * humidity * temp * sunlight * (10**-6)
        return K

    def DMCcycle(RA, T, RH, day, prev) :
        ###
        def raintest(RA, T, RH, day, prev) :
        #RA - precip, T= tas, RH = hurs, day = time_x, prev = previous day DMC
            prev = np.maximum(prev, 0.01) #SAFING FOR NEGATIVE VALUES
            Pe = raineffect(RA)
            Mt = priorduff(prev)
            b_con = [prev > 65, (prev > 33) & (prev <= 65)]
            b_ops = [6.2 * np.log(prev) - 17.2, 14 - (1.3 * np.log(prev))]
            b = np.select(b_con, b_ops, default=100 / (0.5 + 0.3 * prev)) #IF-IFEl-ELSE
            Mfinal = Mt + ((1000 * Pe)/(48.77 + b * Pe))

            Mfinal = np.maximum(Mfinal, 20.0001) #BECAUSE THE 2008 SAYS MOISTURE CAN'T DROP BELOW 20%, DOING THIS TO AVOID 0 ERRORS
            log_inner = Mfinal - 20

            Rt = np.where(Mfinal.values > 20.0,
                         244.72 - 43.43 * np.log(log_inner),
                         0.0)
            Rt = np.maximum(Rt, 0)                    
            K = Ktest(T, RH, day)
            DMC = Rt + 100 * K
            return DMC

        def drytest(T, RH, day, prev) :
            #T= tas, RH = hurs, day = time_x, prev = previous day DMC
            K = Ktest(T, RH, day)
            wholekitandkaboodle = prev + 100 * K
            return wholekitandkaboodle
        duff = np.where(RA.values > 1.5,
                        raintest(RA, T, RH, day, prev),
                        drytest(T, RH, day, prev))
        return duff
    
    
    def DMCshell(RA, T, RH, spinup) :
    #RA = precip, T = tas, RH = relative humidity, spinup = starting DMC value
    #WE CREATE SEPERATE TESTS FOR DMCt-1 AND DMCrt AND THE LOOP PERFORMS A NPWHERE FOR D VALUES
        start_time = time.perf_counter()
        dmcframe = xr.full_like(T, fill_value=np.nan)

        for time_x in range(len(RA)) :
            ra_x = RA.isel(time=time_x)
            t_x = T.isel(time=time_x)
            rh_x = RH.isel(time=time_x)

            run = DMCcycle(ra_x, t_x, rh_x, time_x, spinup)
            dmcframe.values[time_x] = run
            spinup = dmcframe.values[time_x]
        loop_time = timedelta(seconds=time.perf_counter() - start_time)
        print(f"This took us about {loop_time}")
        dmcframe = np.round(dmcframe)
        dmcframe.attrs['units'] = 'degrees'
        DMC = dmcframe.rename('dmc')
        return DMC
    if DMCtrue == True :
        dmcrun = DMCshell(RA, T, RH, 6)
        dmcrun.to_netcdf(f'{date} - {locale}_{now}_{mode}_{model}_DUFF.nc')
        print(f'Duff {model} Success! Saved as an .nc of the ({now}) {model} {mode}\n')
        DMC = dmcrun

    def oldFF(prev):
        if prev < 0 :
            print(prev)
        else : pass
        MtpDenom = 59.5 + prev
        try:
            Mtp = 147.2 * ((101 - prev)/MtpDenom)
        except ZeroDivisionError :
            Mtp = 0
        if Mtp < 0 :
            Mtp = 0
        else :
            Mtp = Mtp
        return Mtp

    def FFcalc(F1, T, RA, H, W):
        e = np.e #EULERS
         #DRY PHASE CANNOT PROCESS NEGATIVES OR SUPERSAT CONDITIONS
        Mtp = oldFF(F1) #IN THE MANUAL AS F0
        #RAINFALL PHASE
        #PRECIP ADJUST
        if RA > 0.5 :
            Pf = RA - 0.5
        else : Pf = RA
        #PREV DAY PRECIP BALANCING  

        if Mtp > 0 and Mtp <= 150 :
            with np.errstate(divide='raise'):
                try :
                    Mrt = Mtp + 42.5 * Pf * (e**(-100/(251-Mtp))) * (1 - (e**(-6.93/Pf)))
                except FloatingPointError:
                    Mrt = 85 #NEUTRAL VALUE IN CASE OF FAILURE FOR NOW
                except ZeroDivisionError:
                    Mrt = F1 #REFER TO PREVDAY VALUE IN CASE OF FAILURE FOR NOW
        else :
            with np.errstate(divide='raise'):
                try :
                    Mrt = Mtp + 42.5 * Pf * (e**(-100/(251-Mtp))) * (1 - (e**(-6.93/Pf))) + 0.0015 * ((Mtp - 150)**2) * (Pf**0.5)

                except FloatingPointError:
                    Mrt = 85 #NEUTRAL VALUE IN CASE OF FAILURE FOR NOW
                except ZeroDivisionError:
                    Mrt = F1 #REFER TO PREVDAY VALUE IN CASE OF FAILURE FOR NOW
        if Mrt >= 250 :
            Mrt = 250
        else : pass
        #DRYING PHASE

        Ed = 0.942 * (H**0.679) + (11 * (e**((H-100)/10)))+0.18*(21.1-T)*(1-e**-(0.115*H)) #ADDED NP ABS TO EXPONENTS TO TRY AND GROUND SCALARS
        #LOG DRY CHECK
        if Mrt > Ed :
            Ko = 0.424 *(1-(H/100)**1.7)+0.0694*(W**0.5)*(1-((100-H)/100)**8)
            Kd = (Ko * 0.581) * (e**(0.0365*T))
            m = Ed + (Mrt - Ed) * (10**(-Kd))
        else : 
            #IN EVENT Ed LARGER THAN PREVIOUS MOISTURE VALUE, WE CALC WET PHASE CONTENT
            Ew = 0.618 * (H**0.753)+10*(e**((H-100)/10))+0.18*(21.1-T)*(1-e**(-0.115*H))
            try :
                Ko = 0.424 * (1-((100-H)/100)**1.7)+0.0694 * (W**0.5) * (1-((100-H)/100)**8)
            except OverflowError :
                Ko = (W**0.5)
            Kd = Ko * 0.581 * (e**(0.0365*T))
            m = Ew - (Ew - Mrt) *(10**-Kd)# DOUBLE CHECK VARIABLES WITH ORIGINAL EQUATION SHEET
            if Ew <= Mrt or Ed >= Mrt :
                m = Mrt
            else : pass
            #REPEATED Ko AND Kd AGAIN FOR READABILITY BUT THE MANUAL HAS THEM AS Ew AND Kw
        FF = 59.5 * ((250-m)/(147.2+m))
        finalanswer = my.floor(FF) #ROUND DOWN OUT OF CAUTION
        return finalanswer

    def finefuel(T, H, R, W, begval) :
        #T = TASMAX / H = HURS / R = PRECIP / W = WIND / begval = FFMC0 /
        start_time = time.perf_counter()
        e = np.e #EULERS
        fineframe = xr.full_like(T, fill_value = np.nan)
        rainframe = xr.full_like(R, fill_value = np.nan)
        for time_x in range(len(W)) :
            W_x = W.isel(time=time_x) 
            T_x = T.isel(time=time_x)
            H_x_u = H.isel(time=time_x)
            R_x = R.isel(time=time_x)
            H_x = np.clip(H_x_u,0,105)

            FFrunner = np.vectorize(FFcalc)
            Fcalc = FFrunner(begval, T_x, R_x, H_x, W_x)
            fineframe.values[time_x] = Fcalc
            begval = fineframe.values[time_x]

        loop_time = timedelta(seconds=time.perf_counter() - start_time)
        print(f"This took us about {loop_time}")
        fineframe = np.round(fineframe)
        fineframe.attrs['units'] = 'degrees'
        ffmc = fineframe.rename('ffmc')
        return ffmc
    if windtrue == True :
        ffmcrun = finefuel(T, RH, RA, W, 85)
        ffmcrun.to_netcdf(f'{date} - {locale}_{now}_{mode}_{model}_FINE.nc')
        print(f'FFMC {model} Success! Saved as an .nc of the ({now}) {model} {mode}')
        FFMC = ffmcrun


    def buildu(DMC, DC) : #LINE 36 IN MANUAL
        #DMC = DUFF MOISTURE CODE, DC = DROUGHT CODE
        numer = (0.8 * DMC * DC)
        denom = (DMC + (0.4 * DC))
        if denom != 0 :
            equation = numer / denom
        else :
            equation = 0
        return equation
    bup = np.vectorize(buildu) 

    def BUI(DMC, DC) :
        #DMC = DUFF MOISTURE CODE, DC = DROUGHT CODE
        start_time = time.perf_counter()
        buildframe = xr.full_like(DC, fill_value=np.nan)
        for time_x in range(len(DMC)) :
            DMC_x = DMC.isel(time=time_x) 
            DC_x = DC.isel(time=time_x) 
            D = DMC_x * 0.4
            zcheck = np.where(DMC_x > 0, #CHECK FOR 0 DMC, WHICH NEGATES BUI
                         bup(D, DC_x),
                        0)
            bui = np.where(zcheck > 199.9, #VALUES ABOVE 200 SHOW EXTREME RISK, WE TRIM THESE FOR SIMPLICITY
                             np.round(zcheck, 0),
                            np.round(zcheck, 2))
            bui = np.round(bui)
            buildframe.values[time_x] = bui
        loop_time = timedelta(seconds=time.perf_counter() - start_time)
        print(f"This took us about {loop_time}")
        buildframe.attrs['units'] = 'units'
        buildupindex = buildframe.rename('bui')
        return buildupindex
    if windtrue == True
        buildthatindex = BUI(DMC, DC)
        buildthatindex.to_netcdf(f'{date} - {locale}_{now}_{mode}_{model}_BUI.nc')
        print(f'BUI {model} Success! Saved  as an .nc of the ({now}) {model} {mode}')
        BUI = buildthatindex
        start_time = time.perf_counter()
        spreadframe = xr.full_like(FFMC, fill_value=0)
        #FIRST WE APPLY THE WIND EFFECT
        e = np.e
        fW = e ** (0.05039 * W) #DOUBLES THE ISI EVERY 14kmh LOSES CERTAINTY AT HIGH WINDS
        #MOISTURE SLOPE FOR FINE FUELS
        try :
            m = 147.2 * ((101 - FFMC)/(59.5 + FFMC))
            m = np.where(m < 0,
                        0,
                        m)
            m = np.where( np.isnan(m),
                        0,
                        m)
        except Exception as e :
            print(e)
        #EMPIRICAL EQUATION FOR FUELS BURN RATE
        try :
            denominator = ((m ** 5.31)/(4.93*(10**7)))
            denom = np.where(np.isnan(denominator),
                            0,
                            denominator)
            empMC= (91.9 * (e ** (-0.1386 * m))) * (1 + denom) #PG 20 IN WAGNER 1987
        except Exception as e :
            print(e)

        final = 0.208 * fW * empMC #* 10
        final_cleaned = np.where(np.isnan(final),
                                 0,
                                 final)
        init = np.round(final_cleaned)
        spreadframe.values = np.round(init)
        loop_time = timedelta(seconds=time.perf_counter() - start_time)
        print(f"FFMC for {model} took about {loop_time}")
        spreadframe.attrs['units'] = 'units'
        initialspreadindex = spreadframe.rename('isi')
        initialspreadindex.to_netcdf(f'{date} - {locale}_{now}_{mode}_{model}_ISI.nc')
        print(f'ISI {model} Success! Saved as an .nc of the ({now}) {model} {mode}')
        isi = initialspreadindex
        I = isi
        B = BUI
        e = np.e

        frame = xr.full_like(isi, fill_value=0)
        #FIRST CONVERT BUI TO WEIGHT OF FUEL CONSUMED
        #THIS IS D IN THE MANUAL
        try :
            D_B = np.where(
                B > 80,
                1000 / (25+ (108.64 * (e ** (-0.023 * B)))),
                ((0.626 * B) ** 0.809)+2
                )
        except Exception as E :
            print(E)

        try :
            BSCALE = 0.1 * I * D_B
        except Exception as e :
            print(e)

        #NOW WE CONVERT TO PURE INTENSITY (S SCALE)
        try :
            S = np.where(BSCALE > 1,
                        np.log(e ** (2.72 * (((0.434*np.log(BSCALE)) ** 0.647)))),
                         BSCALE
                        )
            S = np.round(S, decimals=0)
            #PAGE 24 OF DEVELOPMENT PAPER (WAGNER 1987 STATES ALL COMPONENTS AND END RESULT OF FWI ARE ROUNDED TO WHOLE NUMBERS)
        except Exception as e :
            print(e)

        #NOW WE REPLACE THE FRAME WITH THE FINAL PRODUCT
        frame.values = S
        frame.attrs['units'] = 'degrees'
        index = frame.rename('NeFWI')
        NFWI = index
        NFWI.to_netcdf(f'{date} - {locale}_{now}_{mode}_{model}_NFWI.nc')
        
    FINALRETURN = print(f"{model} finished for ({now}) {mode}")
        
    return FINALRETURN ################################################

locale = 'NE'
local = 'Nebraska'
modellist = #THESE SHOULD BE JUST THE NAMES OF YOUR FILES' MODELS IN STRING FORMAT SUCH AS 'ACCESS-CM2' <<<<<<<<<<
timelist = #THESE SHOULD BE A LIST OF YOUR SCENARIOS IN STRING FORM AS SUCH AS ['Historical', 'ssp585'] <<<<<<<<<<<
stamplist = ['Full', 'NearFuture', 'MidFuture', 'FarFuture'] #SEE DOCUMENTATION FOR REFERENCE ON TIME PERIOD DEFINITIONS
for currentmodel in modellist :
    for timeslice in timelist :
        model = currentmodel
        mode = timeslice
        if mode == 'Historical' :
            now = 'Full'
            if mode == 'Historical' and now == 'Full' :
                timestamp = slice('1950', '2013')
            try : 
                taspath = #PATH TO YOUR TEMPERATURE MAXIMUM NETCDF4 FILES <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
                prpath = #PATH TO YOUR PRECIPITATION NETCDF4 FILES <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
                tasfile = xr.open_dataset(taspath)
                prfile = xr.open_dataset(prpath)
                tas = tasfile['tasmax'].sel(
                    time=timestamp,
                    lat=tasfile['lat'],
                    lon=tasfile['lon'])
                pr = prfile['pr'].sel(
                    time=timestamp,
                    lat=prfile['lat'],
                    lon=prfile['lon'])
                T =  tas 
                RA = pr
                RH = xr.full_like(T, fill_value=np.nan)
                W = xr.full_like(T, fill_value=np.nan)
                try :
                    hurpath = #PATH TO YOUR HUMIDITY NETCDF4 FILES <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
                    hurfile = xr.open_dataset(hurpath)
                    hurs = hurfile['hurs'].sel(
                        time=timestamp,
                        method='nearest',
                        lat=hurfile['lat'],
                        lon=hurfile['lon'])
                    RH = hurs
                    DMCtrue = bool(True)
                except Exception as e :
                    print(f"{model} has no hurs file for {mode}!")
                    DMCtrue = bool(False)
                if DMCtrue == True :
                    try :
                        windpath = #PATH TO YOUR WIND NETCDF4 FILES <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
                        windfile = xr.open_dataset(windpath)
                        sfcwind = windfile['sfcWind'].sel(
                            time=timestamp,
                            method='nearest',
                            lat=windfile['lat'],
                            lon=windfile['lon'])
                        W = sfcwind
                        windtrue = bool(True)
                    except Exception as e :
                        windtrue = bool(False)
                        print(f"{model} has no wind file for {mode}!")
                        
                finalcountdown = ENTIRETHING(T, RH, W, RA, DMCtrue, windtrue)
                print(f"{model} is finished")
            except Exception as woops :
                print(f'{woops}')
                print("SKIPPING TO NEXT RUN ATTEMPT")
                continue
        else:
            for stamp in stamplist :
                now = stamp
                try :
                    if mode == 'ssp585' and now == 'Full' :
                        timestamp = slice('2026', None) #THIS AVOIDS END INDEXING ERRORS
                    elif mode == 'ssp585' and now == 'NearFuture' :
                        timestamp = slice('2026','2050')
                    elif mode == 'ssp585' and now == 'MidFuture' :
                        timestamp = slice('2051', '2075')
                    elif mode == 'ssp585' and now == 'FarFuture' :
                        timestamp = slice('2076', None)
                    else : 
                        print('Invalid Time Frame')
                        print(e)
                        continue
                try : 
                    taspath = #PATH TO YOUR TEMPERATURE MAXIMUM NETCDF4 FILES <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
                    prpath = #PATH TO YOUR PRECIPITATION NETCDF4 FILES <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
                    tasfile = xr.open_dataset(taspath)
                    prfile = xr.open_dataset(prpath)
                    tas = tasfile['tasmax'].sel(
                        time=timestamp,
                        method='nearest',
                        lat=tasfile['lat'],
                        lon=tasfile['lon'])
                    pr = prfile['pr'].sel(
                        time=timestamp,
                        method='nearest',
                        lat=prfile['lat'],
                        lon=prfile['lon'])
                    T =  tas 
                    RA = pr
                    RH = xr.full_like(T, fill_value=np.nan)
                    W = xr.full_like(T, fill_value=np.nan)
                    try :
                        hurpath = #PATH TO YOUR HUMIDITY NETCDF4 FILES <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
                        hurfile = xr.open_dataset(hurpath)
                        hurs = hurfile['hurs'].sel(
                            time=timestamp,
                            method='nearest',
                            lat=hurfile['lat'],
                            lon=hurfile['lon'])
                        RH = hurs
                        DMCtrue = bool(True)
                    except Exception as e :
                        print(f"{model} has no hurs file for {mode}!")
                        DMCtrue = bool(False)
                    if DMCtrue == True :
                        try :
                            windpath = #PATH TO YOUR WIND NETCDF4 FILES <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
                            windfile = xr.open_dataset(windpath)
                            sfcwind = windfile['sfcWind'].sel(
                                time=timestamp,
                                method='nearest',
                                lat=windfile['lat'],
                                lon=windfile['lon'])
                            W = sfcwind
                            windtrue = bool(True)
                        except Exception as e :
                            windtrue = bool(False)
                            print(f"{model} has no wind file for {mode}!") #ADD SIMILAR FOR HURS AND ADD FWI FUNC LOOPS

                    finalcountdown = ENTIRETHING(T, RH, W, RA, DMCtrue, windtrue)
                    print(f"{model} is finished")
                except Exception as woops :
                    print(f'{woops}')
                    print("SKIPPING TO NEXT RUN ATTEMPT")
                    continue


